# SELinux

This Jupyter notebook will allow you to configure SELinux on an active slice.



## Setup and Configuration

In [ ]:
import sys
from pathlib import Path

# Add parent directory to Python path to import modules
repo_root = Path.cwd().parent
# No longer needed - using installed package

print(f"✅ Python path configured")
print(f"   Repository root: {repo_root}")

In [ ]:
# Define YAML directory

YAML_DIR = repo_root / "model"
print(f"✅ YAML directory: {YAML_DIR}")

## Set Slice Name

In [ ]:
# Define your base slice name
slice_name = "slice-ansible-cluster"


## Step 1: Import the FABlib Library


In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
                     
fablib.show_config()

In [ ]:
# Import modules

from fabric_generic_cluster import load_topology_from_yaml_file, SiteTopology
from fabric_generic_cluster import deployment as sd
from fabric_generic_cluster import network_config as snc
from fabric_generic_cluster import ssh_setup as ssh
from fabric_generic_cluster import ansible_setup as ansible
from fabric_generic_cluster import selinux_management as selinux


print("✅ Modules imported successfully")

## Step 2: Check your existing slices



In [ ]:
try:
    for slice in fablib.get_slices():
        print(f"{slice}")
except Exception as e:
    print(f"Exception: {e}")

## Step 3: Observe the Slice's Attributes

### Select the slice 

In [ ]:
try:
    slice = fablib.get_slice(name=slice_name)
    print(f"{slice}")
except Exception as e:
    print(f"Exception: {e}")

### Print the Node List

In [ ]:
try:
    slice = fablib.get_slice(name=slice_name)

    print(f"{slice.list_nodes()}")
except Exception as e:
    print(f"Exception: {e}")

### Print the Node Details (Optional)

In [ ]:
#try:
#    slice = fablib.get_slice(name=slice_name)
#    for node in slice.get_nodes():
#        print(f"{node}")
#except Exception as e:
#    print(f"Exception: {e}")

### Print the Interfaces (Optional)

In [ ]:
#try:
#    slice = fablib.get_slice(name=slice_name)
#    print(f"{slice.list_interfaces()}")
#except Exception as e:
#    print(f"Exception: {e}")

## Step 4: Configure SELinux


In [ ]:
# Site topology YAML file
site_topology_yaml = "../model/m5-v4.yaml"
slice = sd.get_slice(slice_name)

In [ ]:
# Load and validate topology (raises ValidationError if invalid)
try:
    topology = load_topology_from_yaml_file(site_topology_yaml)
    print("✅ Topology loaded and validated successfully!")
    print(f"   Nodes: {len(topology.site_topology_nodes.nodes)}")
    print(f"   Networks: {len(topology.site_topology_networks.networks)}")
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise

### 4.1 Check SELinux

In [ ]:
#
# Check SELinux status on all nodes
#
selinux.check_selinux_status_all_nodes(slice, topology)

In [ ]:
#
# Display SELinux status
#
selinux.display_selinux_summary(slice, topology)

### 4.2 Set SELinux mode

In [ ]:
#
# Set to permissive persistently (survives reboot)
#

selinux.set_selinux_mode_all_nodes(
    slice,
    topology,
    mode=selinux.SELinuxMode.PERMISSIVE,
    persistent=True
)

In [ ]:
#
# Set SELinux to enforcing on specific nodes
#

selinux.set_selinux_mode_all_nodes(
    slice,
    topology,
    mode=selinux.SELinuxMode.ENFORCING,
    persistent=True,
    nodes=["lc-10", "lc-20"]  # Only these nodes
)

In [ ]:
#
# Disable SELinux (requires reboot to take full effect)
# 

selinux.set_selinux_mode_all_nodes(
    slice,
    topology,
    mode=selinux.SELinuxMode.DISABLED,
    persistent=True
)

print("⚠️  Remember to reboot nodes for SELinux disable to take full effect")

In [ ]:
#
# Set permissive mode for OpenStack nodes (convenience function)
#

selinux.set_selinux_permissive_for_openstack(slice, topology)

In [ ]:
#
# Runtime mode change (temporary, until reboot)
#

selinux.set_selinux_mode_all_nodes(
    slice,
    topology,
    mode=selinux.SELinuxMode.PERMISSIVE,
    persistent=False  # Runtime only
)

## Step 5: (Alternative) Set SELinux mode from the topology (model)

In [ ]:
# Ignore topology and force all to permissive
selinux.set_selinux_mode_all_nodes(
    slice,
    topology,
    mode=SELinuxMode.PERMISSIVE,
    persistent=True
)

## Reboot (Optional)

In [ ]:
node_name="lc-1"

In [ ]:
node = slice.get_node(name=node_name)

In [ ]:
reboot = 'sudo reboot'
try:
    print(reboot)
    node.execute(reboot)
    
    slice.wait_ssh(timeout=360,interval=10,progress=True)

    print("Now testing SSH abilites to reconnect...",end="")
    slice.update()
    slice.test_ssh()
    print("Reconnected!")

except Exception as e:
    print(f"Fail: {e}")  

node.config()
print("Done")    